# Phase 4 — Figures

Generates all paper figures from training + LLC metric CSVs.
**No GPU needed.** Runs in < 2 min.

**Kaggle:** Add all Phase 1, 2, 3 outputs as dataset inputs.
Then *Save and Run All* — PDFs land in `results/figures/` and appear as session output.

## Section 0 — Setup

In [ ]:
import os, sys, shutil, subprocess

PLATFORM = "kaggle"   # "kaggle" or "colab"
REPO_URL  = "https://github.com/makataomu/slt-diplomka"

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    REPO_DIR    = "/content/slt"
    PERSIST_DIR = "/content/drive/MyDrive/slt_persist"
    for d in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
else:
    REPO_DIR    = "/kaggle/working/slt"
    PERSIST_DIR = None

if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Pulled latest from GitHub")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Cloned from GitHub")

if PLATFORM == "colab":
    lnk = f"{REPO_DIR}/results"
    if os.path.islink(lnk): os.unlink(lnk)
    elif os.path.isdir(lnk): shutil.rmtree(lnk)
    os.symlink(f"{PERSIST_DIR}/results", lnk)
    print(f"results/ -> {PERSIST_DIR}/results")
else:
    for sub in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{REPO_DIR}/{sub}", exist_ok=True)
    # Restore metrics (and calibration traces) from all dataset inputs
    for inp in sorted(os.listdir("/kaggle/input")):
        prev = f"/kaggle/input/{inp}/slt/results"
        if os.path.exists(prev):
            print(f"Restoring from /kaggle/input/{inp}/ ...")
            for sub in ["metrics"]:
                src, dst = f"{prev}/{sub}", f"{REPO_DIR}/results/{sub}"
                if os.path.exists(src):
                    for item in os.listdir(src):
                        s, d = f"{src}/{item}", f"{dst}/{item}"
                        if not os.path.exists(d):
                            (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
            print("  Done.")

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"\nReady. Platform={PLATFORM} | cwd={os.getcwd()}")

# Show what metrics are available
from pathlib import Path
import pandas as pd
csvs = sorted(Path("results/metrics").glob("ratio_*.csv"))
print(f"\nTraining CSVs:  {len([f for f in csvs if '_llc' not in f.name])}")
print(f"LLC CSVs:       {len([f for f in csvs if '_llc' in f.name])}")


## Section 1 — Install dependencies (matplotlib/pandas only, no GPU)

In [ ]:
%pip install -q matplotlib pandas numpy


## Section 2 — Generate all figures

In [ ]:
!python src/plotting.py

from pathlib import Path
for f in sorted(Path("results/figures").glob("*.pdf")):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")


## Section 3 — Display figures inline

Converts each PDF to PNG for in-notebook preview.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import subprocess

figs = sorted(Path("results/figures").glob("*.pdf"))
if not figs:
    print("No figures yet.")
else:
    for pdf in figs:
        png = pdf.with_suffix(".png")
        subprocess.run(["convert", "-density", "150", str(pdf), str(png)],
                       capture_output=True)
        if png.exists():
            print(f"--- {pdf.name} ---")
            display(Image(filename=str(png)))
        else:
            print(f"{pdf.name} — install ImageMagick to preview inline")
